# Why Multi-Agent? Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: The Overloaded Single Agent

Here is a single agent trying to do everything. It has one massive system prompt and one context window holding research, code, and reviews:

In [ ]:
```typescript

type AgentResult = {

  content: string;

  tokensUsed: number;

  toolCalls: number;

};

async function singleAgentApproach(task: string): Promise<AgentResult> {

  const systemPrompt = `You are a full-stack developer. You must:

1. Research the requirements

2. Write the code

3. Review the code for bugs

4. Write tests

Do ALL of these in a single conversation.`;

  const contextWindow: string[] = [];

  let totalTokens = 0;

  let totalToolCalls = 0;

  const research = await fakeLLMCall(systemPrompt, `Research: ${task}`);

  contextWindow.push(research.output);

  totalTokens += research.tokens;

  totalToolCalls += research.calls;

  const code = await fakeLLMCall(

    systemPrompt,

    `Given this research:\n${contextWindow.join("\n")}\n\nNow write code for: ${task}`

  );

  contextWindow.push(code.output);

  totalTokens += code.tokens;

  totalToolCalls += code.calls;

  const review = await fakeLLMCall(

    systemPrompt,

    `Given all previous context:\n${contextWindow.join("\n")}\n\nReview the code.`

  );

  contextWindow.push(review.output);

  totalTokens += review.tokens;

  totalToolCalls += review.calls;

  return {

    content: contextWindow.join("\n---\n"),

    tokensUsed: totalTokens,

    toolCalls: totalToolCalls,

  };

}

In [ ]:
```

Problems with this approach:

- The context window grows with every stage. By the review step, it contains research notes AND code AND prior reasoning.

- The system prompt is generic. It cannot be tuned for each stage.

- Nothing runs in parallel.

### Step 2: Specialist Agents

Now split it. Each agent gets one job:

In [ ]:
```typescript

type SpecialistAgent = {

  name: string;

  systemPrompt: string;

  run: (input: string) => Promise<AgentResult>;

};

function createSpecialist(name: string, systemPrompt: string): SpecialistAgent {

  return {

    name,

    systemPrompt,

    run: async (input: string) => {

      const result = await fakeLLMCall(systemPrompt, input);

      return {

        content: result.output,

        tokensUsed: result.tokens,

        toolCalls: result.calls,

      };

    },

  };

}

const researcher = createSpecialist(

  "researcher",

  "You are a technical researcher. Read documentation, find patterns, and summarize findings. Output only the facts needed for implementation."

);

const coder = createSpecialist(

  "coder",

  "You are a senior TypeScript developer. Given requirements and research notes, write clean, tested code. Nothing else."

);

const reviewer = createSpecialist(

  "reviewer",

  "You are a code reviewer. Find bugs, security issues, and logic errors. Be specific. Cite line numbers."

);

In [ ]:
```

Each specialist has a focused prompt. Each gets a clean context window with only the input it needs.

### Step 3: Coordinate Through Messages

Wire the specialists together with explicit message passing:

In [ ]:
```typescript

type AgentMessage = {

  from: string;

  to: string;

  content: string;

  timestamp: number;

};

async function multiAgentApproach(task: string): Promise<AgentResult> {

  const messages: AgentMessage[] = [];

  let totalTokens = 0;

  let totalToolCalls = 0;

  const researchResult = await researcher.run(task);

  messages.push({

    from: "researcher",

    to: "coder",

    content: researchResult.content,

    timestamp: Date.now(),

  });

  totalTokens += researchResult.tokensUsed;

  totalToolCalls += researchResult.toolCalls;

  const coderInput = messages

    .filter((m) => m.to === "coder")

    .map((m) => `[From ${m.from}]: ${m.content}`)

    .join("\n");

  const codeResult = await coder.run(coderInput);

  messages.push({

    from: "coder",

    to: "reviewer",

    content: codeResult.content,

    timestamp: Date.now(),

  });

  totalTokens += codeResult.tokensUsed;

  totalToolCalls += codeResult.toolCalls;

  const reviewerInput = messages

    .filter((m) => m.to === "reviewer")

    .map((m) => `[From ${m.from}]: ${m.content}`)

    .join("\n");

  const reviewResult = await reviewer.run(reviewerInput);

  messages.push({

    from: "reviewer",

    to: "orchestrator",

    content: reviewResult.content,

    timestamp: Date.now(),

  });

  totalTokens += reviewResult.tokensUsed;

  totalToolCalls += reviewResult.toolCalls;

  return {

    content: messages.map((m) => `[${m.from} -> ${m.to}]: ${m.content}`).join("\n\n"),

    tokensUsed: totalTokens,

    toolCalls: totalToolCalls,

  };

}

In [ ]:
```

Each agent receives only the messages addressed to it. No context pollution. The researcher's 50k tokens of documentation reading never enter the reviewer's context.

### Step 4: Compare

In [ ]:
```typescript

async function compare() {

  const task = "Build a rate limiter middleware for an Express.js API";

  console.log("=== Single Agent ===");

  const single = await singleAgentApproach(task);

  console.log(`Tokens: ${single.tokensUsed}`);

  console.log(`Tool calls: ${single.toolCalls}`);

  console.log("\n=== Multi-Agent ===");

  const multi = await multiAgentApproach(task);

  console.log(`Tokens: ${multi.tokensUsed}`);

  console.log(`Tool calls: ${multi.toolCalls}`);

}

In [ ]:
```

The multi-agent version uses more total tokens (three agents, three separate LLM calls) but each agent's context stays clean. The quality of each stage improves because the system prompt is specialized.

## Exercises

In [ ]:
1. Add a fourth specialist: a "tester" agent that receives code from the coder and review feedback from the reviewer, then writes tests
2. Modify the pipeline so the reviewer can send feedback back to the coder for a revision loop (max 2 rounds)
3. Convert the sequential pipeline into a fan-out: run the researcher and a "requirements analyzer" agent in parallel, then merge their outputs before passing to the coder